# GPT-2 small (~124M) on Kaggle GPU T4 x2 — PyTorch DDP

Trains a GPT-2 small architecture on a ~1.2B-token slice of FineWeb-Edu using PyTorch DDP across 2 T4 GPUs. Wall time: **~12–18 hours** total — fits in 1–2 Kaggle sessions thanks to time-based checkpointing and auto-resume.

**Setup checklist:**
1. Notebook → right sidebar → **Settings → Accelerator → GPU T4 x2**.
2. Settings → **Persistence → Files only** (so `/kaggle/working/` survives between sessions).
3. Settings → **Internet → On** (needed to download FineWeb-Edu from HuggingFace).

**Quotas:** 30 GPU hours/week on Kaggle (free), 12 hr per session.

## 1. Hardware check

In [ ]:
!nvidia-smi

Expect to see **2 × Tesla T4** (each 16 GB).

## 2. Get the project files

Clone your repo (Option A) or upload the project root and `cd` into it (Option B).

In [ ]:
# Option A — clone (replace with your repo URL):
# !git clone https://github.com/<you>/gpt-2-nano.git /kaggle/working/gpt-2-nano
# %cd /kaggle/working/gpt-2-nano

# Option B — verify files are present in CWD:
import os
for f in ('config.py', 'model.py', 'data.py', 'train.py', 'sample.py'):
    assert os.path.isfile(f), f'missing {f} — run this from the project root'
print('cwd:', os.getcwd())

## 3. Install dependencies

Kaggle ships with a CUDA PyTorch already, so we only need the small extras.

In [ ]:
!pip -q install tiktoken datasets requests

## 4. Set persistent paths

Everything goes in `/kaggle/working/` — that directory survives session restarts when you have **Persistence → Files only** turned on (see Settings).

In [ ]:
import os
PERSIST_DIR = '/kaggle/working/gpt-2-nano-run'
DATA_DIR = f'{PERSIST_DIR}/data'
CKPT_DIR = f'{PERSIST_DIR}/checkpoints'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print('Data:', DATA_DIR)
print('Checkpoints:', CKPT_DIR)

## 5. Tokenize FineWeb-Edu (one-time, ~15–25 min)

Streams ~1.2B tokens from HuggingFace and writes `train.bin` / `val.bin`. Idempotent — safe to rerun, will skip if files already exist with enough tokens.

Doing this **before** the DDP launch keeps two ranks from racing on the download.

In [ ]:
!python data.py --dataset fineweb_edu --data_dir "$DATA_DIR"

## 6. Sanity check the model (1 forward pass on 1 GPU)

In [ ]:
import torch
from config import MODEL_PRESETS
from model import GPT

mcfg = MODEL_PRESETS['small']
model = GPT(mcfg).cuda()
print(f'parameters: {model.num_parameters() / 1e6:.2f}M')
x = torch.randint(0, mcfg.vocab_size, (2, 64), device='cuda')
logits, loss = model(x, x)
print('logits:', logits.shape, 'loss:', loss.item())
del model; torch.cuda.empty_cache()

## 7. Train with DDP across both T4s

`torchrun --nproc_per_node=2` launches 2 ranks, one per GPU. The training loop:
- saves a checkpoint every **3 hours** of wall-clock time (override with `--ckpt_interval_hours`)
- on a rerun, automatically restores the latest checkpoint from `--ckpt_dir` and continues
- logs / saves only from rank 0 to avoid mangled output

Effective batch tokens per step: `batch_size(8) × grad_accum(32) × world_size(2) × block_size(1024) ≈ 524k`, matching the original GPT-2 small recipe.

**You can interrupt and rerun this cell — it picks up from the latest checkpoint.**

In [ ]:
!torchrun --standalone --nproc_per_node=2 train.py \
    --preset small \
    --dataset fineweb_edu \
    --data_dir "$DATA_DIR" \
    --ckpt_dir "$CKPT_DIR" \
    --ckpt_interval_hours 3.0 \
    --max_iters 6000

## 8. Sample from the trained model

In [ ]:
!python sample.py \
    --ckpt_dir "$CKPT_DIR" \
    --prompt "The history of mathematics begins" \
    --max_new_tokens 300 \
    --temperature 0.8 \
    --top_k 200

## 9. Resuming after a session timeout

If your 12 h Kaggle session ends mid-training, the latest checkpoint is on disk in `/kaggle/working/gpt-2-nano-run/checkpoints/`. To continue:

1. **Click "Save Version" → "Quick Save"** before disconnect (or right after you notice the session is about to die). This snapshots `/kaggle/working/` as a notebook output.
2. **In your next session**, attach the previous version's output as a Kaggle Dataset (Notebook → +Add data → Notebook output) and copy it back into `/kaggle/working/`:
    ```
    !cp -r /kaggle/input/<your-prev-output>/checkpoints /kaggle/working/gpt-2-nano-run/
    ```
3. Rerun cells 1, 3, 4, 7. The trainer detects the existing checkpoint and resumes from there.

Alternatively, if Persistence → Files only is on, `/kaggle/working/` may survive between sessions on its own — test by listing the directory at the start of the next session.

Worst-case work lost: ~3 h (the checkpoint interval). Tighten with `--ckpt_interval_hours 1.0` if you want fewer at-risk hours, at the cost of more disk write time.

## Speed-tuning knobs (if T4 x2 throughput looks low)

- **Lower `--gradient_accumulation_steps`** — fewer micro-steps means less DDP all-reduce overhead per macro-step, but smaller effective batch (worse final loss).
- **Increase `--batch_size`** — only if `nvidia-smi` shows < 14 GB used. Try `--batch_size 12`.
- **`--compile`** — `torch.compile` sometimes gives 1.2–1.5x on T4. Sometimes also crashes. Try at your own risk.
- **Reduce `--max_iters`** — 4000 iters (~2.1B tokens trained) gets you ~95% of the val-loss benefit at ~67% of the time. Diminishing returns past 6000 on this data slice.